In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
try:
    import neuralprophet
except:
    !pip install neuralprophet

<hr style="border: solid 3px blue;">

# Introduction

> The data used is a compilation of passengers from January 1949 to December 1960. We want to predict the time series through the RNN model made of simple LTSM for the data.

![](https://insightimi.files.wordpress.com/2020/07/on-de793_201909_g_20190830121038.gif)

Picture Credit: https://insightimi.files.wordpress.com

**What is time series analysis?**
> Time series analysis comprises methods for analyzing time series data in order to extract meaningful statistics and other characteristics of the data. Time series forecasting is the use of a model to predict future values based on previously observed values. While regression analysis is often employed in such a way as to test relationships between one or more different time series, this type of analysis is not usually called "time series analysis", which refers in particular to relationships between different points in time within a single series. Interrupted time series analysis is used to detect changes in the evolution of a time series from before to after some intervention which may affect the underlying variable.

Ref: https://en.wikipedia.org/wiki/Time_series

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

import torch

In [ ]:
def plot_decompose(decompose_result):
    fig, (ax1,ax2,ax3,ax4) = plt.subplots(4,1,figsize=(12,20))
    decompose_result.observed.plot(legend=False,ax=ax1,fontsize = 20,grid=True,linewidth = 3)
    ax1.set_ylabel("Observed",fontsize = 20)
    decompose_result.trend.plot(legend=False,ax=ax2,fontsize = 20,grid=True,linewidth = 3)
    ax2.set_ylabel("Trend",fontsize = 20)
    decompose_result.seasonal.plot(legend=False,ax=ax3,fontsize = 20,grid=True,linewidth = 3)
    ax3.set_ylabel("Seasonal",fontsize = 20)
    decompose_result.resid.plot(legend=False,ax=ax4,fontsize = 20,grid=True,linewidth = 3)
    ax4.set_ylabel("Residual",fontsize = 20)

# EDA (Exploratory Data Analysis)

In [ ]:
flight_data = pd.read_csv('/kaggle/input/flight/flight.csv')
cm = sns.light_palette("green", as_cmap=True)
flight_data.head(20).style.background_gradient(cmap=cm)

In [ ]:
print(flight_data.describe())
print('-'*40)
print(flight_data.tail())

In [ ]:
flight_data.shape

In [ ]:
flight_data.info()

In [ ]:
plt.figure(figsize=(12,5))
plt.title('Month vs Passenger',fontsize = 20)
plt.ylabel('Total Passengers',fontsize = 20)
plt.xlabel('Months',fontsize = 20)
plt.grid(True)
plt.autoscale(axis='x',tight=True)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.plot(flight_data['passengers'])

If you look at the picture above, you can see the periodicity by season. Let's analyze it a bit more using seasonal_decompose,

In [ ]:
flight_data['passengers']

Let's decompose time series data into Trend, Seasonality, and Residual through time series decomposition.

In [ ]:
import statsmodels
import statsmodels.api as sm  
from statsmodels.tsa.stattools import acf  
from statsmodels.tsa.stattools import pacf
from statsmodels.tsa.seasonal import seasonal_decompose

decomposition = seasonal_decompose(flight_data['passengers'], period=12) 
plot_decompose(decomposition)

As expected, seasonal periodicity was observed. Let's see if our predictive model can learn this periodicity and make a prediction. Also, if you look at the trend line, you can see that the number of customers is gradually increasing.

# Preprocessing

In [ ]:
all_data = flight_data['passengers'].values.astype(float)
print(all_data)

In [ ]:
test_data_size = 12

train_data = all_data[:-test_data_size]
test_data = all_data[-test_data_size:]

In [ ]:
print(len(train_data))
print(len(test_data))

Perform Min-Max scaling. 

In this case, the distribution of the dataset is maintained through linear transformation. You can refer to the notebook below for more details.

[preprocessing-linear-nonlinear-scaling](https://www.kaggle.com/ohseokkim/preprocessing-linear-nonlinear-scaling)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(-1, 1))
train_data_normalized = scaler.fit_transform(train_data .reshape(-1, 1))

In [ ]:
train_data_normalized = torch.FloatTensor(train_data_normalized).view(-1)

# Utility Function

> This function returns a tuple by transforming the raw input data into sequence data to fit training. The number of passengers traveling in the first 12 months predicts the number of passengers in 13 months.
> 
> The first value in tuple: sequence of passengers in 12 months (=features)
> 
> The second value of the tuple: the number of passengers predicted as the number of passengers in 12 months (=target)

In [ ]:
train_window = 12

In [ ]:
def create_inout_sequences(input_data, window):
    inout_seq = []
    L = len(input_data)
    for i in range(L-window):
        train_seq = input_data[i:i+window]
        train_label = input_data[i+window:i+window+1]
        inout_seq.append((train_seq ,train_label))
    return inout_seq

In [ ]:
train_inout_seq = create_inout_sequences(train_data_normalized, train_window)

In [ ]:
train_inout_seq[:5]

<hr style="border: solid 3px blue;">

# Modeling using LTSM

![](https://miro.medium.com/max/1400/1*goJVQs-p9kgLODFNyhl9zA.gif)

Picture Credit: https://miro.medium.com

> Long short-term memory (LSTM) is an artificial recurrent neural network (RNN) architecture used in the field of deep learning. Unlike standard feedforward neural networks, LSTM has feedback connections. It can process not only single data points (such as images), but also entire sequences of data (such as speech or video). For example, LSTM is applicable to tasks such as unsegmented, connected handwriting recognition, speech recognition and anomaly detection in network traffic or IDSs (intrusion detection systems).
> 
> A common LSTM unit is composed of a cell, an input gate, an output gate and a forget gate. The cell remembers values over arbitrary time intervals and the three gates regulate the flow of information into and out of the cell.
> 
> LSTM networks are well-suited to classifying, processing and making predictions based on time series data, since there can be lags of unknown duration between important events in a time series. LSTMs were developed to deal with the vanishing gradient problem that can be encountered when training traditional RNNs. Relative insensitivity to gap length is an advantage of LSTM over RNNs, hidden Markov models and other sequence learning methods in numerous applications.

Ref: https://en.wikipedia.org/wiki/Long_short-term_memory

Modeling is done using a simple LTSM layer.

* Input_size: It corresponds to the number of input sequences. The sequence length is 12, but there is only 1 value per month, i.e. total number of passengers, so the input size is 1.
* Hidden_layer_size: Specifies the number of hidden layers.
* Output_size: The output size is 1 because the number of items in the output predicts the number of passengers in the next month.

In [ ]:
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

In [ ]:
class LSTM(nn.Module):
    def __init__(self, input_size=1, hidden_layer_size=128, num_layers=2, output_size=1):
        super().__init__()
        self.hidden_layer_size = hidden_layer_size
        self.lstm = nn.LSTM(input_size, hidden_layer_size, num_layers=num_layers)
        self.linear = nn.Linear(hidden_layer_size, output_size)

    def forward(self, input_seq):
        lstm_out, _ = self.lstm(input_seq.view(len(input_seq) ,1, -1))
        predictions = self.linear(lstm_out[:,-1,:])
        return predictions[-1]

In [ ]:
model = LSTM()
loss_function = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

In [ ]:
print(model)

# Training

In [ ]:
epochs = 500

for i in range(epochs):
    for seq, labels in train_inout_seq:
        optimizer.zero_grad()
 
        y_pred = model(seq)

        single_loss = loss_function(y_pred, labels)
        single_loss.backward()
        optimizer.step()

    if i%25 == 1:
        print(f'epoch: {i:3} loss: {single_loss.item():10.8f}')
print(f'epoch: {i:3} loss: {single_loss.item():10.8f}')

# Predicting

The test set contains passenger data for the last 12 months. The model is trained to make predictions using sequence length 12. Let's predict the last 12 months of data.

In [ ]:
fut_pred = 12

test_inputs = train_data_normalized[-train_window:].tolist()
print(test_inputs)

The test set is run 12 iterations. At the end of the iteration, the test_inputs list contains 24 entries. The last 12 items are the predicted values for the test set.

In [ ]:
model.eval()

for i in range(fut_pred):
    seq = torch.FloatTensor(test_inputs[-train_window:])
    with torch.no_grad():
        test_inputs.append(model(seq).item())

Check the last 12 predictions.

In [ ]:
test_inputs[fut_pred:]

# Converting to real values
Since we normalized the dataset for training, the predicted values are also normalized. We need to transform the normalized predicted values into the actual predicted values. Use the inverse_transform of the min/max scaler object you used to normalize the data set to transform it to its original value.

In [ ]:
actual_predictions = scaler.inverse_transform(np.array(test_inputs[train_window:] ).reshape(-1, 1))
print(actual_predictions)

In [ ]:
x = np.arange(132, 144, 1)
print(x)

# Checking results
The prediction of the LSTM is indicated by the orange line. Although the results are not exact, it is possible to spot an upward trend based on fluctuations in the total number of passengers traveling over the past 12 months. Better performance can be achieved by using a larger number of epochs and a larger number of neurons in the LSTM layer.

In [ ]:
plt.figure(figsize=(12,5))
plt.title('Month vs Passenger',fontsize = 20)
plt.ylabel('Total Passengers',fontsize = 20)
plt.xlabel('Months',fontsize = 20)
plt.grid(True)
plt.autoscale(axis='x',tight=True)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.plot(flight_data['passengers'])
plt.plot(x,actual_predictions)

In [ ]:
plt.title('Month vs Passenger')
plt.ylabel('Total Passengers')
plt.grid(True)
plt.autoscale(axis='x', tight=True)

plt.plot(flight_data['passengers'][-train_window:])
plt.plot(x,actual_predictions)
plt.show()

# Time series analysis with predicted results

Check whether the predicted results are learned while preserving the trend, seasonality, and residual of the timer series that the original time series has.

In [ ]:
flight_data['passengers'][:-train_window]
train_df = pd.DataFrame(flight_data['passengers'][:-train_window])
actual_df = pd.DataFrame(actual_predictions)
actual_df.columns = ['passengers']
new_predict = pd.concat([train_df,actual_df]).reset_index(drop=True)

In [ ]:
plt.figure(figsize=(12,5))
plt.title('Month vs Passenger',fontsize = 20)
plt.ylabel('Total Passengers',fontsize = 20)
plt.xlabel('Months',fontsize = 20)
plt.grid(True)
plt.autoscale(axis='x',tight=True)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.plot(new_predict)
plt.plot(flight_data['passengers'])

**Let's do a seasonal decomposition analysis with the predicted results.**

In [ ]:
decomposition = seasonal_decompose(new_predict, period=12) 
plot_decompose(decomposition)

Looking at the figure above, it can be confirmed that, despite the simple model, it is predicted while well preserving the trend, seasonality, and residual.

<hr style="border: solid 3px blue;">

# Predicting by Prophet

![](https://miro.medium.com/max/1400/0*CHhERBPfiUJJDJo1.gif)

Picture Credit: https://miro.medium.com

Let's predict using prophet.

> Prophet is a procedure for forecasting time series data based on an additive model where non-linear trends are fit with yearly, weekly, and daily seasonality, plus holiday effects. **It works best with time series that have strong seasonal effects and several seasons of historical data.** Prophet is robust to missing data and shifts in the trend, and typically handles outliers well.

Ref: https://facebook.github.io/prophet/


## Preprocessing

If you try to train using Prophet, you need to change the column and data type according to the conditions required by prophet.

In [ ]:
df = flight_data.copy()

In [ ]:
month2int = {
    'January': 1,
    'February': 2,
    'March': 3,
    'April': 4,
    'May': 5,
    'June': 6,
    'July': 7,
    'August': 8,
    'September': 9,
    'October': 10,
    'November': 11,
    'December': 12
}
df['month'] = df['month'].map(month2int)

In [ ]:
df['day'] = 1
df['ds'] = pd.to_datetime(df[['year','month','day']])
df_new = df.drop(columns=['year','month','day'])
df_new.rename(columns={"passengers": "y"},inplace=True)
df_new.head()

## Trainging

In [ ]:
from fbprophet import Prophet 
m = Prophet()
m.fit(df_new)

## Predicting

In [ ]:
future = m.make_future_dataframe(periods=500)
forecast = m.predict(future)

## Checking results

In [ ]:
fig2 = m.plot_components(forecast)

In [ ]:
from fbprophet.plot import plot_plotly, plot_components_plotly

plot_plotly(m, forecast)

In [ ]:
plot_components_plotly(m, forecast)

<hr style="border: solid 3px blue;">

# NeuralProphet

![](https://miro.medium.com/max/1838/1*g_clFunR6GbWU1rL2iGugA.png)

Picture Credit: https://miro.medium.com


> NeuralProphet is a Neural Network based PyTorch implementation of a user-friendly time series forecasting tool for practitioners. This is heavily inspired by Prophet, which is the popular forecasting tool developed by Facebook. NeuralProphet is developed in a fully modular architecture which makes it scalable to add any additional components in the future. Our vision is to develop a simple to use forecasting tool for users while retaining the original objectives of Prophet such as interpretability, configurability and providing much more such as the automatic differencing capabilities by using PyTorch as the backend.

Ref: https://neuralprophet.com

In [ ]:
from neuralprophet import NeuralProphet

In [ ]:
m = NeuralProphet()
metrics = m.fit(df_new, freq="M")
forecast = m.predict(df_new)

In [ ]:
fig_forecast = m.plot(forecast)
fig_components = m.plot_components(forecast)
fig_model = m.plot_parameters()

# Predicting by NeuralProphet

In [ ]:
future = m.make_future_dataframe(df_new, periods=50, n_historic_predictions=len(df_new)-50)
forecast = m.predict(future)
fig_forecast = m.plot(forecast)
fig_components = m.plot_components(forecast)
fig_model = m.plot_parameters()

<hr style="border: solid 3px blue;">

# Conclusion

The same dataset was predicted using simple RNN ,prophet and NeuralProphet. All of these methods seem to be predictable. However, it is expected that the predictive power will decrease in datasets that do not have constant trend and seasonality. In other words, it seems that an appropriate method should be selected based on the domain knowledge of the corresponding dataset.
